<a href="https://colab.research.google.com/github/G0rav/machine_learning_explained_visually_free/blob/main/02-differentiation/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Differentiation: The Calculus You Actually Need for Machine Learning

**[Watch the video](https://www.youtube.com/@school_whool)** · video 2 of *Machine Learning Explained
Visually*.

Everything from the video, runnable. Three layers:

1. **Follow along** — every derivation in the video, in code, in the same order.
2. **Experiment** — break the results on purpose and find out why they break.
3. **Challenge** — derive the next piece yourself before the next video.

Nothing here imports from anywhere. `numpy` is required; `sympy` and
`matplotlib` are optional and every cell that uses them says so. Run it top to
bottom.

Every section asserts the number the video put on screen. If a check ever fails,
the video and this notebook have drifted apart and one of them is wrong.

---

# Layer 1 — Follow along

In [ ]:
import numpy as np

PASSED = 0
def check(label, got, want, tol=1e-9):
    """Assert a value the video states, and say so."""
    global PASSED
    got_f, want_f = float(got), float(want)
    assert abs(got_f - want_f) <= tol, f"{label}: {got_f!r} != {want_f!r}"
    PASSED += 1
    print(f"ok   {label:44s} {got_f:.6g}")

## Part 1 — Why derivatives run machine learning

The video opens by re-running video 1: a bad line, a loss that measures how far
it is from the points, and a step that makes the loss smaller.

The loss is the sum of squared residuals — for each point, the gap between the
value it has and the value the line predicts, squared, added up.

In [ ]:
FIT_X = np.array([0.6, 1.4, 2.1, 2.9, 3.6, 4.3, 5.1, 5.8])
FIT_Y = np.array([1.15, 1.60, 1.95, 2.65, 2.90, 3.55, 3.80, 4.45])

def fit_loss(m, c):
    """Sum of squared residuals: the number the opening watches come down."""
    return float(np.sum((FIT_Y - (m * FIT_X + c)) ** 2))

BAD_M, BAD_C = 0.30, 2.30          # the deliberately bad line the video opens on
check("part 1  loss at the bad line", fit_loss(BAD_M, BAD_C), 4.6551, 1e-3)

### How video 1 found the direction: nudge, and measure

Change one parameter by a little, see what the loss did, divide. That ratio is
an *estimate* of the derivative — it is the thing this whole video replaces.

The video nudges the slope `m` by `0.05` and the loss gets **worse**, which is
how you learn the direction: step the other way.

In [ ]:
NUDGE = 0.05
before = fit_loss(BAD_M, BAD_C)
after  = fit_loss(BAD_M + NUDGE, BAD_C)
ratio  = (after - before) / NUDGE

check("part 1  loss after nudging m", after, 5.4814, 1e-3)
check("part 1  the finite difference", ratio, 16.526, 1e-3)
print(f"\nthe loss got worse ({before:.2f} -> {after:.2f}), so m should go the other way")

Two things to notice, because the rest of the video is about both:

* the ratio depends on the step size `0.05` — a different step gives a different
  answer, and none of them is *the* rate of change;
* it costs one extra evaluation of the loss per parameter. A model with a
  million parameters costs a million extra evaluations, per step.

The derivative fixes both.

## Part 2 — What a derivative is measuring

Two points on a curve, the change in `y` over the change in `x`. The video uses
these two, and reads the four numbers off the picture.

In [ ]:
def f_demo(x):
    """The curve parts 2 and 3 are drawn on."""
    return 0.05 * np.asarray(x, float) ** 3 - 0.42 * np.asarray(x, float) ** 2 \
           + 1.30 * np.asarray(x, float) + 1.30

def df_demo(x):
    return 0.15 * np.asarray(x, float) ** 2 - 0.84 * np.asarray(x, float) + 1.30

X1, X2 = 1.3, 5.1
y1, y2 = float(f_demo(X1)), float(f_demo(X2))
dy, dx = y2 - y1, X2 - X1

check("part 2  y1 = f(x1)", y1, 2.3901, 1e-4)
check("part 2  y2 = f(x2)", y2, 3.6383, 1e-4)
check("part 2  delta y", dy, 1.2483, 1e-4)
check("part 2  delta x", dx, 3.8, 1e-9)
check("part 2  delta y / delta x", dy / dx, 0.3285, 1e-4)

### And that ratio is tan(theta)

The vertical side over the horizontal side of a right-angled triangle *is* the
tangent of the angle at its base. That is not an analogy, it is the definition,
and part 4 uses it to read the sign of a derivative off a picture.

In [ ]:
theta = np.arctan2(dy, dx)
check("part 2  tan(theta) is the same ratio", np.tan(theta), dy / dx, 1e-12)
print(f"theta = {np.degrees(theta):.2f} degrees")

## Part 3 — Letting the gap go to zero

The ratio depends on which two points you picked. Bring the second one in
towards the first and watch what it settles on. The video walks this schedule.

In [ ]:
def secant(f, a, b):
    return (float(f(b)) - float(f(a))) / (b - a)

WALK = (5.10, 4.20, 3.30, 2.50, 1.90, 1.55, 1.40, 1.34)
print(f"{'x2':>6}  {'delta x':>8}  {'ratio':>8}")
for x2 in WALK:
    print(f"{x2:6.2f}  {x2 - X1:8.2f}  {secant(f_demo, X1, x2):8.4f}")

check("part 3  the ratio at the far point", secant(f_demo, X1, WALK[0]), 0.3285, 1e-4)
check("part 3  the tangent it approaches", df_demo(X1), 0.4615, 1e-4)

The ratio never *is* `0.4615` for any two distinct points — it approaches it.
That is the whole content of the limit: the derivative is the number the ratio
gets arbitrarily close to, not a ratio you can compute directly.

In [ ]:
for h in (1e-1, 1e-3, 1e-5, 1e-7):
    est = secant(f_demo, X1, X1 + h)
    print(f"h = {h:<8g} ratio = {est:.10f}   error = {abs(est - df_demo(X1)):.2e}")

## Part 4 — Reading the sign

`dy/dx = tan(theta)`, so the sign of the derivative is the sign of `tan` of the
angle the tangent makes with the x axis:

| theta | tan(theta) | the curve is |
|---|---|---|
| between 0 and 90 degrees | positive | going up |
| exactly 0 | zero | flat, for an instant |
| exactly 90 degrees | undefined | vertical |
| between 90 and 180 degrees | negative | coming down |

The video walks one point along a hump and reads all four off it.

In [ ]:
def f_hump(x):
    return 4.30 - 0.42 * (np.asarray(x, float) - 3.0) ** 2

def df_hump(x):
    return -0.84 * (np.asarray(x, float) - 3.0)

PEAK = 3.0
check("part 4  the peak is where the slope is zero", df_hump(PEAK), 0.0, 1e-12)

print(f"{'x':>5}  {'slope':>8}  {'theta':>8}  sign")
for x in (1.10, 2.40, PEAK, 3.60, 4.90):
    slope = float(df_hump(x))
    theta = np.degrees(np.arctan(slope)) % 180.0
    if abs(slope) < 1e-9:
        theta, sign = 0.0, "zero — the top"
    else:
        sign = "positive — rising" if slope > 0 else "negative — falling"
    print(f"{x:5.2f}  {slope:8.3f}  {theta:7.1f}d  {sign}")

## Part 5 — The rules

One derivative done from the definition, in full, and then the results.

    f'(x) = lim(h -> 0) [ f(x + h) - f(x) ] / h

For `f(x) = x^2`:

    f(x + h)        = (x + h)^2 = x^2 + 2xh + h^2
    f(x + h) - f(x) =            2xh + h^2
    divided by h    =            2x  + h
    let h -> 0      =            2x

Note the step the video is careful about: we divide by `h` **while `h` is still
nonzero**, and only then let it go. Nothing is ever divided by zero.

In [ ]:
def x_squared_ratio(x, h):
    """The difference quotient for x^2, simplified by hand to 2x + h."""
    return 2.0 * x + h

x = 1.10
for h in (1.0, 0.1, 0.01, 0.001, 0.0):
    print(f"h = {h:<7g} (f(x+h) - f(x)) / h = {x_squared_ratio(x, h):.4f}")
check("part 5  d/dx x^2 at x = 1.1", x_squared_ratio(x, 0.0), 2.2, 1e-12)

# and the same thing numerically, without the hand simplification
raw = (( (x + 1e-6) ** 2 ) - x ** 2) / 1e-6
check("part 5  the numerical quotient agrees", raw, 2.2, 1e-5)

### The rules the video collects

| rule | derivative |
|---|---|
| `x^n` | `n x^(n-1)` |
| a constant `c` | `0` |
| `c f(x)` | `c f'(x)` |
| `log x` | `1 / x` |
| `e^x` | `e^x` |
| `f(x) + g(x)` | `f'(x) + g'(x)` |

Each one checked against a numerical derivative, which is the honest way to
believe a rule you have not derived yourself.

In [ ]:
def numeric_d(f, x, h=1e-6):
    """A central difference: more accurate than the one-sided one above."""
    return (float(f(x + h)) - float(f(x - h))) / (2.0 * h)

x = 2.0
cases = [
    ("x^n, n = 5",   lambda t: t ** 5,          5 * x ** 4),
    ("a constant",   lambda t: 7.0 + 0 * t,     0.0),
    ("3 x^4",        lambda t: 3.0 * t ** 4,    12 * x ** 3),
    ("log x",        lambda t: np.log(t),       1.0 / x),
    ("e^x",          lambda t: np.exp(t),       float(np.exp(x))),
    ("x^2 + 5x",     lambda t: t ** 2 + 5 * t,  2 * x + 5),
]
for label, f, by_rule in cases:
    check(f"part 5  {label}", numeric_d(f, x), by_rule, 1e-4)

## Part 6 — The chain rule

    d/dx f(g(x)) = df/dg  *  dg/dx

The video differentiates `(a - bx)^2` with it, and the notebook does what the
video does not have room for: it checks the answer a second way, by expanding
the bracket first and differentiating that. Two routes, one answer.

In [ ]:
A, B = 5.0, 2.0

def by_chain_rule(x):
    """df/dg * dg/dx, with g = a - bx and f = g^2."""
    g      = A - B * x
    df_dg  = 2.0 * g          # d/dg of g^2
    dg_dx  = -B               # d/dx of a - bx
    return df_dg * dg_dx

def by_expanding(x):
    """(a - bx)^2 = a^2 - 2abx + b^2 x^2, then differentiate term by term."""
    return -2.0 * A * B + 2.0 * B ** 2 * x

for x in (0.0, 1.5, 3.0):
    check(f"part 6  chain rule = expansion at x = {x}",
          by_chain_rule(x), by_expanding(x), 1e-12)
    check(f"part 6  and both match a numeric derivative at x = {x}",
          numeric_d(lambda t: (A - B * t) ** 2, x), by_chain_rule(x), 1e-5)

print(f"\nd/dx (a - bx)^2 = -2b(a - bx),  at x = 1.5 that is {by_chain_rule(1.5):.1f}")

## Part 7 — Maxima and minima

At a maximum and at a minimum the tangent is flat, so `df/dx = 0`. That turns a
question about *shape* into an equation you can solve — which is the property
the rest of the video runs on.

The video's curve falls, turns up, rises, turns down. Its two turning points are
found by solving `f'(x) = 0`.

In [ ]:
def f_turns(x):
    t = np.asarray(x, float)
    return -0.25 * t ** 3 + 1.50 * t ** 2 - 2.00 * t + 3.00

def df_turns(x):
    t = np.asarray(x, float)
    return -0.75 * t ** 2 + 3.00 * t - 2.00

# f' is a quadratic, so solve it exactly rather than searching.
a, b, c = -0.75, 3.00, -2.00
disc = b ** 2 - 4 * a * c
roots = sorted([(-b + np.sqrt(disc)) / (2 * a), (-b - np.sqrt(disc)) / (2 * a)])

check("part 7  the minimum", roots[0], 0.8453, 1e-4)
check("part 7  the maximum", roots[1], 3.1547, 1e-4)
for r in roots:
    check(f"part 7  the slope at {r:.4f} is zero", df_turns(r), 0.0, 1e-12)

## Part 8 — Actually finding one

The worked example, end to end: differentiate, set to zero, solve, then check
which kind of point you found.

    f(x)   = x^2 - 3x + 2
    df/dx  = 2x - 3
    2x - 3 = 0   ->   x = 3/2 = 1.5

In [ ]:
def f_quad(x):
    t = np.asarray(x, float)
    return t ** 2 - 3.0 * t + 2.0

def df_quad(x):
    return 2.0 * np.asarray(x, float) - 3.0

x_star = 1.5
check("part 8  the derivative is zero at x*", df_quad(x_star), 0.0, 1e-12)
check("part 8  x* solves 2x - 3 = 0", (3.0 / 2.0), x_star, 1e-12)

### Which kind is it? Check *both* sides

A zero slope happens at a maximum and at a minimum, so solving the equation does
not finish the job. The video looks at a point either side.

One side is not enough: a point can have zero slope with the curve higher on one
side and lower on the other — `x^3` at the origin does exactly that — and such a
point is neither a maximum nor a minimum. That case is checked below too.

In [ ]:
f_star  = float(f_quad(x_star))
f_below = float(f_quad(1.0))
f_above = float(f_quad(2.0))

check("part 8  f(1.5)", f_star, -0.25, 1e-12)
check("part 8  f(1)",   f_below, 0.0, 1e-12)
check("part 8  f(2)",   f_above, 0.0, 1e-12)

print(f"f(1) = {f_below}, f(1.5) = {f_star}, f(2) = {f_above}")
print("both neighbours are higher, so x = 1.5 is a minimum")

def classify(f, x, delta=0.5):
    """Minimum, maximum or neither, from the two neighbours."""
    here, left, right = float(f(x)), float(f(x - delta)), float(f(x + delta))
    if left > here and right > here:
        return "minimum"
    if left < here and right < here:
        return "maximum"
    return "neither"

check("part 8  classified as a minimum", classify(f_quad, x_star) == "minimum", True)
check("part 8  x^3 at 0 is neither", classify(lambda t: np.asarray(t, float) ** 3, 0.0) == "neither", True)
print("\nx^3 at the origin: slope zero, lower on the left, higher on the right — neither")

## Part 9 — Local and global

A function can have several minima at different depths. Setting the derivative
to zero finds **all** of them and cannot tell you which is the deepest — that is
the problem gradient descent inherits, and the reason the next video exists.

In [ ]:
def f_twodip(x):
    t = np.asarray(x, float)
    return 0.16 * t ** 4 - 1.40 * t ** 3 + 3.60 * t ** 2 - 2.60 * t + 3.20

def df_twodip(x):
    t = np.asarray(x, float)
    return 0.64 * t ** 3 - 4.20 * t ** 2 + 7.20 * t - 2.60

def bisect(g, lo, hi, steps=200):
    """A sign change bracketed and halved: no library, no guessing."""
    flo = float(g(lo))
    for _ in range(steps):
        mid = 0.5 * (lo + hi)
        if flo * float(g(mid)) <= 0:
            hi = mid
        else:
            lo, flo = mid, float(g(mid))
    return 0.5 * (lo + hi)

stationary = [bisect(df_twodip, a, b) for a, b in ((0.0, 1.0), (1.5, 2.5), (3.5, 4.5))]
check("part 9  the local minimum", stationary[0], 0.4914, 1e-4)
check("part 9  the maximum between them", stationary[1], 2.0619, 1e-4)
check("part 9  the global minimum", stationary[2], 4.0092, 1e-4)

depths = [float(f_twodip(s)) for s in stationary]
check("part 9  the shallow dip", depths[0], 2.6349, 1e-4)
check("part 9  the deep dip", depths[2], 1.7598, 1e-4)

for s, d in zip(stationary, depths):
    print(f"x = {s:.4f}   f = {d:.4f}   slope = {float(df_twodip(s)):+.1e}   {classify(f_twodip, s, 0.3)}")
print("\nall three satisfy df/dx = 0 — the equation cannot rank them")

## Part 10 — When you can't solve it

    f(x) = log(1 + e^(ax))

Differentiate it with the chain rule and set the result to zero:

    df/dx = a e^(ax) / (1 + e^(ax)) = 0

A fraction is zero only when its numerator is. The denominator cannot help —
`1 + e^(ax)` is always greater than 1 — and `a` is a nonzero constant, so it
comes down to `e^(ax) = 0`, which never happens. **The equation has no
solution**, and the cell below watches `e^(ax)` shrink towards zero without ever
arriving.

In [ ]:
A_CONST = 1.0

def softplus(x, a=A_CONST):
    """log(1 + e^(ax)), written so a large ax does not overflow."""
    return np.logaddexp(0.0, a * np.asarray(x, float))

def softplus_d(x, a=A_CONST):
    """a e^(ax) / (1 + e^(ax)) — the logistic function, times a."""
    return a / (1.0 + np.exp(-a * np.asarray(x, float)))

check("part 10 the derivative at x = -2", softplus_d(-2.0), 0.1192029220, 1e-9)
check("part 10 the derivative at x = 0", softplus_d(0.0), 0.5, 1e-12)
check("part 10 the derivative at x = 1.5", softplus_d(1.5), 0.8175744762, 1e-9)

print("e^(ax) as x goes far negative:")
for x in (0.0, -5.0, -10.0, -20.0, -40.0, -100.0):
    print(f"  x = {x:7.1f}   e^(ax) = {np.exp(x):.3e}   derivative = {float(softplus_d(x)):.3e}")

In [ ]:
# The claim the video makes: the derivative is never zero, anywhere.
xs = np.linspace(-500.0, 500.0, 200001)
d = softplus_d(xs)
check("part 10 the derivative is positive everywhere", bool(np.all(d > 0.0)), True)
check("part 10 its smallest value on this range is still above zero",
      bool(d.min() > 0.0), True)
print(f"smallest derivative sampled: {d.min():.3e} at x = {xs[d.argmin()]:.1f}")
print("it approaches 0 as ax -> -infinity, and -infinity is not a value of x")

So the method that worked in part 8 — set the derivative to zero and solve —
is not always available. That is what the next video is for.

## Part 11 — When x is a vector

`x` in machine learning is a vector, so the derivative becomes a vector of
partial derivatives, one per component, written `grad f`.

For `y = a^T x`, each partial keeps only the term containing its own component:

    d/dx_1 [ a_1 x_1 + a_2 x_2 + ... ] = a_1

so `grad(a^T x) = a`.

In [ ]:
A_VEC = np.array([3.0, -2.0, 5.0])

def y_of(x, a=A_VEC):
    return float(np.dot(a, np.asarray(x, float)))

def grad_by_hand(a=A_VEC):
    """One partial at a time, the way part 11 takes them."""
    return np.array([a[i] for i in range(len(a))], dtype=float)

def grad_numeric(f, x, h=1e-6):
    """Every partial by a central difference, to check the hand answer."""
    x = np.asarray(x, float)
    out = np.zeros_like(x)
    for i in range(x.size):
        step = np.zeros_like(x)
        step[i] = h
        out[i] = (f(x + step) - f(x - step)) / (2.0 * h)
    return out

point = np.array([0.7, -1.3, 2.0])
g_hand = grad_by_hand()
g_num = grad_numeric(y_of, point)

check("part 11 grad(a^T x) = a", float(np.max(np.abs(g_hand - A_VEC))), 0.0, 1e-12)
check("part 11 and a numeric gradient agrees", float(np.max(np.abs(g_hand - g_num))), 0.0, 1e-6)
check("part 11 one entry per component of x", len(g_hand), len(point))
print(f"grad = {g_hand}, and it does not depend on where you evaluate it")

### The two sentences the video ends on

1. The derivative is the slope of the tangent, and it tells you the rate of
   change at a point.
2. At a maximum or a minimum, that derivative is zero.

Everything else was either a rule for computing the first, or a consequence of
the second.

In [ ]:
print(f"Layer 1 complete — {PASSED} checks passed, all of them numbers the video states.")

---

# Layer 2 — Experiment

The video says a two-point ratio only ever *approximates* the derivative. Here
is where that approximation breaks, and why — the experiment the video does not
have room for.

## Smaller steps are better, until they are catastrophic

The error of a central difference falls like `h^2`, so shrinking `h` helps —
right up until subtracting two nearly equal floats destroys the answer. There is
a best `h`, and it is nowhere near the smallest one.

In [ ]:
truth = float(df_demo(X1))
print(f"{'h':>10}  {'estimate':>14}  {'error':>10}")
best_h, best_err = None, float("inf")
for k in range(1, 17):
    h = 10.0 ** (-k)
    est = (float(f_demo(X1 + h)) - float(f_demo(X1 - h))) / (2.0 * h)
    err = abs(est - truth)
    if err < best_err:
        best_h, best_err = h, err
    print(f"{h:10.0e}  {est:14.10f}  {err:10.2e}")
print(f"\nbest h was {best_h:.0e} with error {best_err:.2e}")
print("below that, floating point subtraction loses more than the method gains")

## The cost of nudging, in a model's worth of parameters

Part 1's estimate costs one extra evaluation *per parameter*. The derivative
costs one pass, whatever the count. This is the practical reason the video
exists.

In [ ]:
for d in (2, 100, 1_000_000, 175_000_000_000):
    print(f"{d:>15,} parameters -> {d:>15,} extra evaluations per step, by nudging")
print("\n...versus one backward pass, by differentiating.")

## Where the classification test fails

`classify` compares the two neighbours at a fixed distance, so it is only
trustworthy while that distance stays inside the feature being measured. Run it
on all three stationary points at a range of distances and see which answers
survive — the cell reports what it finds rather than what this paragraph
expects.

In [ ]:
deltas = (0.10, 0.30, 0.80, 1.50, 2.50)
names = ("the local minimum", "the maximum between", "the global minimum")
wrong = []
for name, s, truth in zip(names, stationary, ("minimum", "maximum", "minimum")):
    got = [classify(f_twodip, s, d) for d in deltas]
    marks = "  ".join(f"{d:.2f}:{g}" for d, g in zip(deltas, got))
    print(f"{name:22s} {marks}")
    wrong += [(name, d, g) for d, g in zip(deltas, got) if g != truth]

print()
for name, d, got in wrong:
    print(f"wrong at delta = {d}: {name} reported as '{got}'")
print(f"\n{len(wrong)} of {len(deltas) * 3} answers are wrong, all at the widest step.")
print("the maximum is the one that breaks first: it sits between two dips, so a")
print("wide enough step lands past both of them and the comparison stops meaning")
print("anything. 'A point nearby' has to be *near* -- near enough to stay inside")
print("the feature you are asking about.")

## Check the hand-derived rules with a second engine

Rule 8 of this series: derive it by hand, then check it with a library. `sympy`
is optional — skip this cell if you do not have it.

In [ ]:
try:
    import sympy as sp
except ImportError:
    print("sympy not installed — skipping (every result above was hand-derived anyway)")
else:
    x = sp.symbols("x")
    a, b = sp.symbols("a b", positive=True)
    pairs = [
        ("x^2 - 3x + 2", x**2 - 3*x + 2, 2*x - 3),
        ("(a - bx)^2",   (a - b*x)**2,   -2*b*(a - b*x)),
        ("log(1 + e^x)", sp.log(1 + sp.exp(x)), sp.exp(x)/(1 + sp.exp(x))),
    ]
    for label, expr, by_hand in pairs:
        same = sp.simplify(sp.diff(expr, x) - by_hand) == 0
        print(f"{'ok  ' if same else 'FAIL'} {label:16s} sympy agrees with the video")
        assert same, label
    root = sp.solve(sp.diff(x**2 - 3*x + 2, x), x)
    print(f"\nsympy solves 2x - 3 = 0 as x = {root[0]} — the video's 1.5")
    print("and for log(1 + e^x):", sp.solve(sp.diff(sp.log(1 + sp.exp(x)), x), x),
          "-- an empty list: no solution, exactly as part 10 argues")

## See it (optional)

`matplotlib` is optional. This draws the two pictures the video spends longest
on: the secant closing onto the tangent, and a curve above its own derivative.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed — skipping the plots")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    xs = np.linspace(0.0, 6.2, 400)
    ax1.plot(xs, f_demo(xs), color="0.3")
    for x2, shade in zip((5.10, 3.30, 1.90, 1.34), ("0.75", "0.6", "0.45", "0.2")):
        m = secant(f_demo, X1, x2)
        ax1.plot(xs, f_demo(X1) + m * (xs - X1), color=shade, lw=1)
    ax1.plot(X1, f_demo(X1), "o", color="black")
    ax1.set_title("part 3: the secant closing onto the tangent")

    xs2 = np.linspace(-0.3, 4.9, 400)
    ax2.plot(xs2, f_twodip(xs2), label="f")
    ax2.plot(xs2, df_twodip(xs2), label="f'", lw=1)
    ax2.axhline(0.0, color="0.8", lw=1)
    for s in stationary:
        ax2.plot(s, 0.0, "o", ms=4)
    ax2.legend()
    ax2.set_title("part 9: f' crosses zero at every turning point")

    plt.tight_layout()
    plt.show()

---

# Layer 3 — Challenge

Four exercises, in the order the next video will need them. Each has a check you
can run against your own answer.

**1. Derive `d/dx x^3` from the definition.** Expand `(x + h)^3`, subtract
`x^3`, divide by `h`, let `h` go to zero. Fill in `my_x3_derivative` and the
check will tell you whether it matches.

In [ ]:
def my_x3_derivative(x):
    # replace with your own answer, e.g. return 3 * x ** 2
    return None

if my_x3_derivative(2.0) is None:
    print("not attempted yet — edit my_x3_derivative above")
else:
    for x in (-1.5, 0.0, 2.0, 4.0):
        got = float(my_x3_derivative(x))
        want = numeric_d(lambda t: t ** 3, x)
        print(f"{'ok  ' if abs(got - want) < 1e-4 else 'FAIL'} x = {x:5.1f}  "
              f"yours {got:9.4f}   numeric {want:9.4f}")

**2. Find the minimum of `f(x) = 2x^2 - 8x + 3` the way part 8 does.**
Differentiate, set to zero, solve, then check both sides.

In [ ]:
def my_stationary_point():
    # replace with your own answer
    return None

if my_stationary_point() is None:
    print("not attempted yet — edit my_stationary_point above")
else:
    f = lambda t: 2.0 * np.asarray(t, float) ** 2 - 8.0 * t + 3.0
    x_you = float(my_stationary_point())
    print(f"slope there: {numeric_d(f, x_you):+.2e}  (should be 0)")
    print(f"kind: {classify(f, x_you, 0.5)}  (should be minimum)")
    print(f"{'ok' if abs(numeric_d(f, x_you)) < 1e-4 else 'not there yet'}")

**3. The gradient of a quadratic form.** For `y = x^T x` (the sum of squares of
the components), work out `grad y` by taking one partial at a time, then check
it numerically.

In [ ]:
def my_grad_of_x_dot_x(x):
    # replace with your own answer
    return None

point3 = np.array([0.5, -2.0, 3.0])
if my_grad_of_x_dot_x(point3) is None:
    print("not attempted yet — edit my_grad_of_x_dot_x above")
else:
    want = grad_numeric(lambda v: float(np.dot(v, v)), point3)
    got = np.asarray(my_grad_of_x_dot_x(point3), float)
    print(f"yours   {got}")
    print(f"numeric {want}")
    print("ok" if np.max(np.abs(got - want)) < 1e-5 else "not there yet")

**4. Before the next video.** Gradient descent takes a step *against* the
derivative: `x <- x - lr * f'(x)`. Run it on the two-dip curve from two
different starting points and watch where each one ends up. Nothing here is
graded — the point is to see the problem part 9 describes, before the next
video solves it.

In [ ]:
def descend(df, x0, lr=0.05, steps=200):
    x = float(x0)
    for _ in range(steps):
        x -= lr * float(df(x))
    return x

for start in (0.0, 1.0, 2.5, 4.5):
    end = descend(df_twodip, start)
    which = "global" if abs(end - stationary[2]) < 0.05 else "local"
    print(f"start {start:4.1f} -> {end:6.3f}   f = {float(f_twodip(end)):.4f}   ({which})")
print("\nsame rule, same curve, different answers — that is the whole problem.")